In [7]:
!git clone https://github.com/prava241/flash-attention.git
%cd flash-attention

Cloning into 'flash-attention'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 17 (delta 2), reused 16 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), done.
Resolving deltas: 100% (2/2), done.
/content/flash-attention/flash-attention


In [ ]:
!git pull origin main

In [ ]:
import torch
import numpy as np

N = 512
D = 64

Q = torch.randn(N, D, device="cuda")
K = torch.randn(N, D, device="cuda")
V = torch.randn(N, D, device="cuda")

Q.cpu().numpy().astype(np.float32).tofile("data/q.bin")
K.cpu().numpy().astype(np.float32).tofile("data/k.bin")
V.cpu().numpy().astype(np.float32).tofile("data/v.bin")

TypeError: randn(): argument 'size' failed to unpack the object at pos 2 with error "type must be tuple of ints,but got Tensor"

In [ ]:
# Saves output to data/output.bin, reports runtime
# TODO: Profiles for bottlenecks
!nvcc -O3 src/main.cu src/kernels.cu -o attention_test

In [ ]:
import math

start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)
start.record()

S = Q @ K.T
S = S / math.sqrt(D)
P = torch.softmax(S, dim=-1)
O = P @ V

end.record()
torch.cuda.synchronize()
elapsed_ms = start.elapsed_time(end)

print(f"pytorch_attention runtime: {elapsed_ms:.3f} ms")

reference = O.cpu().numpy()
output = np.fromfile("data/output.bin", dtype=np.float32).reshape(N, D)

print("Max abs error:", np.max(np.abs(reference - output)))
print("Mean abs error:", np.mean(np.abs(reference - output)))
print("All close:", np.allclose(reference, output, atol=1e-5, rtol=1e-5))